<a href="https://colab.research.google.com/github/qsardor/GoogleColabProjects/blob/main/MOSS_TTS_Nano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 MOSS-TTS-Nano Auto-Launcher
#@markdown ---
#@markdown **Source:** [https://github.com/OpenMOSS/MOSS-TTS-Nano](https://github.com/OpenMOSS/MOSS-TTS-Nano)
#@markdown **Creator:** Q.Sardor
#@markdown **Other Colab Projects:** [https://github.com/qsardor/GoogleColabProjects](https://github.com/qsardor/GoogleColabProjects)
#@markdown ---

import os
import subprocess
import sys
from IPython.display import clear_output

# Suppress TF/CUDA noise globally
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore'

print("⏳ Fetching repository...")
if not os.path.exists("MOSS-TTS-Nano"):
    subprocess.run("git clone https://github.com/OpenMOSS/MOSS-TTS-Nano.git", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.chdir("MOSS-TTS-Nano")

print("🗑️ Purging conflicting vision packages...")
subprocess.run("pip uninstall -y torchvision torchtext", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("📦 Compiling dependencies (This takes about 60 seconds)...")
subprocess.run("pip install -r requirements.txt && pip install -e . && pip install gradio", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

clear_output()
print("✅ Build Stable. System Ready.")

import gradio as gr
import torch
import warnings
warnings.filterwarnings("ignore")

from moss_tts_nano_runtime import NanoTTSService, DEFAULT_AUDIO_TOKENIZER_PATH, DEFAULT_CHECKPOINT_PATH

target_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Hardware Pipeline Assigned: [{target_device.upper()}]")
print("🚀 Initializing Gradio Tunnel...")

engine = NanoTTSService(
    checkpoint_path=DEFAULT_CHECKPOINT_PATH,
    audio_tokenizer_path=DEFAULT_AUDIO_TOKENIZER_PATH,
    device=target_device,
    dtype="float32"
)
engine.get_model()

app = gr.Interface(
    fn=lambda text_input, audio_ref: engine.synthesize(
        text=text_input,
        mode="voice_clone",
        prompt_audio_path=audio_ref,
        do_sample=True
    ).get("audio_path"),
    inputs=[
        gr.Textbox(label="Target Text", placeholder="Enter the text to synthesize..."),
        gr.Audio(type="filepath", label="Reference Voice (Upload .wav)")
    ],
    outputs=gr.Audio(label="Generated Audio"),
    title="MOSS-TTS-Nano",
    theme="default"
)

app.launch(share=True, quiet=True)